# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Basil-Maqbool/flyrank-internship-assignment1"
REPO_DIR = "flyrank-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    target_dir = f"{REPO_DIR}/work/notebooks"
    if os.path.basename(os.getcwd()) != "notebooks":
        os.chdir(target_dir)
print("Ready! Current Working Directory:", os.getcwd())

Ready! Current Working Directory: c:\Users\Basil\Desktop\flyrank-internship-assignment1\work\notebooks


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Key traffic/engagement columns — almost always heavy-tailed
traffic_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'pageviews_90d']

print("=== DISTRIBUTIONS OF KEY TRAFFIC COLUMNS ===")
print("(Heavy tails are expected — a few giants, long tail of small values)")
print()
for col in traffic_cols:
    vals = df[col].dropna()
    print(f"{col}:")
    print(f"  Mean: {vals.mean():,.0f}  Median: {vals.median():,.0f}  Max: {vals.max():,.0f}")
    print(f"  90th pct: {vals.quantile(0.9):,.0f}  Ratio (mean/median): {vals.mean()/vals.median():.1f}x")
    print()

=== DISTRIBUTIONS OF KEY TRAFFIC COLUMNS ===
(Heavy tails are expected — a few giants, long tail of small values)

impressions_90d:
  Mean: 5,200  Median: 731  Max: 517,715
  90th pct: 12,136  Ratio (mean/median): 7.1x

clicks_90d:
  Mean: 16  Median: 1  Max: 4,178
  90th pct: 32  Ratio (mean/median): 16.1x

sessions_90d:
  Mean: 37  Median: 7  Max: 4,345
  90th pct: 88  Ratio (mean/median): 5.3x

pageviews_90d:
  Mean: 50  Median: 8  Max: 5,998
  90th pct: 116  Ratio (mean/median): 6.2x



In [3]:
# Distributions of content properties
print("=== CONTENT PROPERTIES ===")
for col in ['content_age_days', 'days_since_last_update', 'word_count']:
    vals = df[col].dropna()
    print(f"{col}:")
    print(f"  Mean: {vals.mean():.0f}  Median: {vals.median():.0f}  Min: {vals.min():.0f}  Max: {vals.max():.0f}")
    print(f"  Missing: {df[col].isna().sum():,} ({df[col].isna().mean():.1%})")
    print()

# Position column — note the avg_position=0 trap
print("avg_position:")
print(f"  Mean: {df['avg_position'].mean():.1f}  Median: {df['avg_position'].median():.1f}")
print(f"  avg_position=0 rows: {(df['avg_position']==0).sum():,} (means 'no data', not rank zero)")

=== CONTENT PROPERTIES ===
content_age_days:
  Mean: 256  Median: 236  Min: 90  Max: 564
  Missing: 0 (0.0%)

days_since_last_update:
  Mean: 46  Median: 20  Min: 1  Max: 373
  Missing: 0 (0.0%)

word_count:
  Mean: 3108  Median: 2877  Min: 8  Max: 9546
  Missing: 7,699 (25.7%)

avg_position:
  Mean: 16.3  Median: 10.8
  avg_position=0 rows: 1,205 (means 'no data', not rank zero)


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [4]:
# Create the target
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# === SIGNAL TEST 1 ===
print("=== SIGNAL TEST 1: Stale pages decline more ===")
print("Claim: Pages not updated in 180+ days are more likely to be declining.")
print()

stale = df[df['days_since_last_update'] >= 180]
fresh = df[df['days_since_last_update'] < 180]

print(f"Stale (>=180d): n={len(stale):,}, decline rate={stale['is_declining_label'].mean():.1%}")
print(f"Fresh (<180d):  n={len(fresh):,}, decline rate={fresh['is_declining_label'].mean():.1%}")
print(f"Difference: {(stale['is_declining_label'].mean() - fresh['is_declining_label'].mean()):.1%}")
print()
print("Verdict: OPPOSITE — stale pages decline LESS (47.1% vs 54.2%).")
print("The small stale group (n=174) has a lower decline rate than the fresh group (n=29,826).")
print("Possible explanation: stale pages that survive may be evergreen content that holds rankings.")

=== SIGNAL TEST 1: Stale pages decline more ===
Claim: Pages not updated in 180+ days are more likely to be declining.

Stale (>=180d): n=174, decline rate=47.1%
Fresh (<180d):  n=29,826, decline rate=54.2%
Difference: -7.1%

Verdict: OPPOSITE — stale pages decline LESS (47.1% vs 54.2%).
The small stale group (n=174) has a lower decline rate than the fresh group (n=29,826).
Possible explanation: stale pages that survive may be evergreen content that holds rankings.


In [5]:
# === SIGNAL TEST 2 ===
print("=== SIGNAL TEST 2: High-impression pages decline less ===")
print("Claim: Pages with high impressions are stable (they rank well, so why would they decline?).")
print()

# Use log1p for heavy-tailed impressions, then bucket by quantiles
df['impression_bucket'] = pd.qcut(df['impressions_90d'], q=5, labels=['Q1-low', 'Q2', 'Q3', 'Q4', 'Q5-high'], duplicates='drop')

impression_decline = df.groupby('impression_bucket', observed=True).agg(
    n=('is_declining_label', 'count'),
    decline_rate=('is_declining_label', 'mean')
).round(3)

display(impression_decline)
print()
print("Verdict: MIXED — relationship is non-linear. Very low and very high impression pages")
print("both show different patterns; the middle is noisy. Not a simple monotonic relationship.")

=== SIGNAL TEST 2: High-impression pages decline less ===
Claim: Pages with high impressions are stable (they rank well, so why would they decline?).



,n,decline_rate
impression_bucket,,
Q1-low,6041,0.325
Q2,5964,0.603
Q3,5997,0.605
Q4,5998,0.633
Q5-high,6000,0.546



Verdict: MIXED — relationship is non-linear. Very low and very high impression pages
both show different patterns; the middle is noisy. Not a simple monotonic relationship.


In [6]:
# === SIGNAL TEST 3 ===
print("=== SIGNAL TEST 3: Older content declines more ===")
print("Claim: Content age is positively associated with decline.")
print()

df['age_bucket'] = pd.cut(df['content_age_days'], bins=[0, 180, 365, 545, 1000], 
                          labels=['<180d', '180-365d', '365-545d', '545d+'])

age_decline = df.groupby('age_bucket', observed=True).agg(
    n=('is_declining_label', 'count'),
    decline_rate=('is_declining_label', 'mean')
).round(3)

display(age_decline)
print()
print("Verdict: OPPOSITE -- decline rate DECREASES with age for the main body.")
print("<180d: 62.7% -> 180-365d: 51.5% -> 365-545d: 41.9%. Only the tiny 545d+ bucket (n=110)")
print("jumps to 82.7%, but n=110 is too small to trust. The dominant trend is inverted.")

=== SIGNAL TEST 3: Older content declines more ===
Claim: Content age is positively associated with decline.



,n,decline_rate
age_bucket,,
<180d,12272,0.627
180-365d,11368,0.515
365-545d,6250,0.419
545d+,110,0.827



Verdict: OPPOSITE -- decline rate DECREASES with age for the main body.
<180d: 62.7% -> 180-365d: 51.5% -> 365-545d: 41.9%. Only the tiny 545d+ bucket (n=110)
jumps to 82.7%, but n=110 is too small to trust. The dominant trend is inverted.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [7]:
# FlyRank's baseline rule: stale + visible = flag for review
# Test: Does the combination of staleness AND visibility actually predict decline?

print("=== FLAG-LINKED TEST: Stale + Visible = Needs Refresh? ===")
print("FlyRank's rule flags pages as: stale (>=180d update) AND visible (>=500 impressions).")
print("Question: Do flagged pages actually decline more than non-flagged pages?")
print()

flagged = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]
not_flagged = df[~((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500))]

print(f"Flagged (stale+visible): n={len(flagged):,}, decline rate={flagged['is_declining_label'].mean():.1%}")
print(f"Not flagged:              n={len(not_flagged):,}, decline rate={not_flagged['is_declining_label'].mean():.1%}")
print(f"Difference: {(flagged['is_declining_label'].mean() - not_flagged['is_declining_label'].mean()):.1%}")
print()
print("Verdict: INSUFFICIENT DATA — flagged group has only n=17, below the ~50-row")
print("minimum needed for a trustworthy rate. The 94.1% decline rate is striking but")
print("unreliable with such a small sample. Cannot confirm or reject the flag.")

=== FLAG-LINKED TEST: Stale + Visible = Needs Refresh? ===
FlyRank's rule flags pages as: stale (>=180d update) AND visible (>=500 impressions).
Question: Do flagged pages actually decline more than non-flagged pages?

Flagged (stale+visible): n=17, decline rate=94.1%
Not flagged:              n=29,983, decline rate=54.2%
Difference: 39.9%

Verdict: INSUFFICIENT DATA — flagged group has only n=17, below the ~50-row
minimum needed for a trustworthy rate. The 94.1% decline rate is striking but
unreliable with such a small sample. Cannot confirm or reject the flag.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [8]:
print("=== PRACTICAL IMPLICATIONS ===")
print()
print("1. Staleness alone does NOT predict decline — stale pages actually decline less.")
print("   This suggests evergreen content that hasn't been updated may still perform well.")
print()
print("2. Content age shows an inverted relationship: younger pages decline MORE, not less.")
print("   This could reflect newer content not yet established in rankings, or higher churn.")
print()
print("3. The stale+visible flag needs more data (n=17 is too small to evaluate).")
print("   A machine learning model can test these interactions with proper sample sizes.")

=== PRACTICAL IMPLICATIONS ===

1. Staleness alone does NOT predict decline — stale pages actually decline less.
   This suggests evergreen content that hasn't been updated may still perform well.

2. Content age shows an inverted relationship: younger pages decline MORE, not less.
   This could reflect newer content not yet established in rankings, or higher churn.

3. The stale+visible flag needs more data (n=17 is too small to evaluate).
   A machine learning model can test these interactions with proper sample sizes.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.